In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau 

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 10       # phase 1: train head only, base frozen
EPOCHS_FINETUNE = 15   # phase 2: unfreeze top layers of Xception, fine-tune
DATA_DIR = "../data"
MODEL_OUT = "deepfake_model.h5"
CLASS_NAMES = ["fake", "real"]  # alphabetical order matches flow_from_directory default


In [3]:
#DATA PIPELINE

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=CLASS_NAMES,
    shuffle=True,
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "valid"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=CLASS_NAMES,
    shuffle=False,
)

Found 8000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [11]:
def build_model():
    base_model=Xception(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base_model.trainable = False  # freeze for phase 1

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=base_model.input , outputs=output)
    return model,base_model

if __name__ == "__main__":
    model, base_model = build_model()


Phase 1 = Feature Extraction (Xception fully frozen, only head trains)

In [12]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss = "binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],

)

callbacks_phase1 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

model.fit(
    train_generator , 
    epochs=EPOCHS_HEAD,
    validation_data=val_generator,
    callbacks=callbacks_phase1,
)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.6595 - auc: 0.7130 - loss: 0.6798 - precision: 0.6601 - recall: 0.6575 - val_accuracy: 0.6970 - val_auc: 0.7943 - val_loss: 0.5959 - val_precision: 0.6401 - val_recall: 0.9000 - learning_rate: 0.0010
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 291s 1s/step - accuracy: 0.7004 - auc: 0.7722 - loss: 0.5818 - precision: 0.7020 - recall: 0.6963 - val_accuracy: 0.7440 - val_auc: 0.8234 - val_loss: 0.5164 - val_precision: 0.7268 - val_recall: 0.7820 - learning_rate: 0.0010
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 292s 1s/step - accuracy: 0.7259 - auc: 0.7989 - loss: 0.5473 - precision: 0.7320 - recall: 0.7128 - val_accuracy: 0.7370 - val_auc: 0.8248 - val_loss: 0.5161 - val_precision: 0.7375 - val_recall: 0.7360 - learning_rate: 0.0010
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 292s 1s/step - accuracy: 0.7401 - auc: 0.8160 - loss: 0.5259 - precision: 0.7476 - recall: 0.7250 - val_accuracy: 0.7450 - val_auc: 0.8351 - val_loss: 0.5059 -

Phase 2: fine-tuning top Xception layers

In [13]:
base_model.trainble = True 
#freeze all but the last 30 layers not 

for layer in base_model.layers[:-30]:
    layer.trainable=False

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

callbacks_phase2 = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    ModelCheckpoint(MODEL_OUT, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
]

model.fit(
    train_generator,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_generator,
    callbacks=callbacks_phase2,
)        

Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 996ms/step - accuracy: 0.7860 - auc: 0.8646 - loss: 0.4608 - precision: 0.7895 - recall: 0.7800
Epoch 1: val_auc improved from None to 0.87465, saving model to deepfake_model.h5



Epoch 1: finished saving model to deepfake_model.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.7860 - auc: 0.8646 - loss: 0.4608 - precision: 0.7895 - recall: 0.7800 - val_accuracy: 0.7880 - val_auc: 0.8746 - val_loss: 0.4534 - val_precision: 0.7727 - val_recall: 0.8160 - learning_rate: 1.0000e-05
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7928 - auc: 0.8729 - loss: 0.4465 - precision: 0.8007 - recall: 0.7795
Epoch 2: val_auc did not improve from 0.87465
250/250 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.7928 - auc: 0.8729 - loss: 0.4465 - precision: 0.8007 - recall: 0.7795 - val_accuracy: 0.7880 - val_auc: 0.8736 - val_loss: 0.4530 - val_precision: 0.7759 - val_recall: 0.8100 - learning_rate: 1.0000e-05
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7887 - auc: 0.8739 - loss: 0.4433 - precision: 0.7969 - recall: 0.7750
Epoch 3: val_auc did not improve from 0.87465
250/250 ━━━━━━━━━━━━━━━━━━━━ 287s 1s/step - accuracy: 0.7887 - 


Epoch 10: finished saving model to deepfake_model.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1573s 6s/step - accuracy: 0.7937 - auc: 0.8719 - loss: 0.4485 - precision: 0.8046 - recall: 0.7760 - val_accuracy: 0.7880 - val_auc: 0.8750 - val_loss: 0.4515 - val_precision: 0.7759 - val_recall: 0.8100 - learning_rate: 1.0000e-05
Epoch 11/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 990ms/step - accuracy: 0.8001 - auc: 0.8785 - loss: 0.4381 - precision: 0.8057 - recall: 0.7910
Epoch 11: val_auc did not improve from 0.87501
250/250 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.8001 - auc: 0.8785 - loss: 0.4381 - precision: 0.8057 - recall: 0.7910 - val_accuracy: 0.7900 - val_auc: 0.8749 - val_loss: 0.4506 - val_precision: 0.7799 - val_recall: 0.8080 - learning_rate: 5.0000e-06
Epoch 12/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 986ms/step - accuracy: 0.7924 - auc: 0.8751 - loss: 0.4433 - precision: 0.7950 - recall: 0.7880
Epoch 12: val_auc did not improve from 0.87501
250/250 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accurac

In [14]:
import os
os.path.exists("deepfake_model.h5")

True

In [15]:
print(os.getcwd())

/Users/harshsisodiya678/TOPS/Sessions/deepfake-detector/model


In [16]:
test_generator = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=CLASS_NAMES,
    shuffle=False,
)

results = model.evaluate(test_generator)
print("Final Test Results:")
for name, val in zip(model.metrics_names, results):
    print(f"  {name}: {val:.4f}")

Found 1000 images belonging to 2 classes.
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 981ms/step - accuracy: 0.7740 - auc: 0.8590 - loss: 0.4727 - precision: 0.7500 - recall: 0.8220
Final Test Results:
  loss: 0.4727
  compile_metrics: 0.7740


In [17]:
print(results)

[0.47271856665611267, 0.7739999890327454, 0.85903000831604, 0.75, 0.8220000267028809]
